# Hidden Markov Models - Viterbi Algorithm
---

The structure of a Hidden Markov Model is specified by:
* a set of __hidden states__: the categories
* an __alphabet__ of observable symbols: the kinds of values the data shows
* the __observation sequence__: the track measured with (T) windows
* the __hidden path__: one state for every window

The set of numbers that create the model are:
* __initial probabilities__, the chance of starting in each state
* __transition probabilities__, the chance of moving from each state to each state
* __emission probabilities__, the chance of each state emitting each symbol

HMMs utilize Markov assumptions to move through the data and provide an output with the given probabilities.
1. **Assumption 1**: The next state depends only on the current state
2. **Assumption 2**: Each observation depends only on the state of its own window


## The Viterbi Algorithm: *The Single Most Likely Path*
---

The Viterbi algorithm focuses on "how likely is an observation sequence (*evaluation*). The question is precise:

> "Given the observations, what *single sequence* of hidden states has the highest joint probability with them?"

The Viterbi path is the sequence of individually most probable states at each position. The number of possible state paths grows as K^N for K states and N observations, so Viterbi uses dynamic programming.


## How Our Code Addresses these Questions
---
### *Decoding with the Viterbi algorithm*

1. "The most probable path" and "the most probable state at each position" sound alike. Are they the same thing? Can you construct a tiny example where the two disagree?`
    * They are not the same thing. The "most probable path" is represented in our ```traceback()``` method, where the Viterbi algorithm finds the single best *whole* path.
    * **A tiny example:**
    ```
   init_probs = {"green": 0.6, "orange": 0.4}
   trans_probs = {"green": {"green": 0.45, "orange": 0.55},
                 "orange": {"green": 0.3, "orange": 0.7}}
   emit_probs = {"green": {"clover": 0.25, "marigold": 0.75}}
   observation = "marigold marigold"
   ```
   * The possible paths
   | Path | Probability |
   | --- | --- |
   | green green | 0.6 x 0.45 = 0.27|
   | green orange | 0.6 x 0.55 = 0.33|
   | orange green | 0.4 x 0.3 = 0.12|
   | orange orange | 0.4 x 0.7 = 0.28|
   * Viterbi picks clover orange with a probability of 0.33.

2. The Viterbi recurrence has the same shape as the Smith-Waterman recurrence from Project 05. Line them up: what plays the role of the score, what plays the role of the gap, and what plays the role of the traceback?
   * Score: ```scores[position][state]```, the best log probability of being a certain state at that position
   * Gap Penalty: the log transition probability
   * Match/ Mismatch score: the log emission probability
   * Traceback: the ```traceback()``` method

3. How many probabilities must be considered to fill one column of the dynamic programming matrix? Does that match the states O(N x K^2) time?`
    * One column takes K^2.
    * With N columns, N X K^2
    * The three nested ```for``` loops of position, current and preceding states match the time.

### *Why log space?*

1. Is log a monotonic function? If a > b, what can you say about log a and log b? What, then, should happen to the max when you move to log space?
    * log is a monotonic function.
        * If a > b, then log a > log b.
    * When moving to log space, multiplying probabilities turns into adding logarithmic probabilities, with the final result being in log.
2. What is log 0? What does Python or numpy give you and how will that value behave when you add it to other numbers or compare it with them?
    * log 0 = -inf
        * ```math.log(0)``` gives an error.
    * When adding it to other numbers or comparing with other numbers, the output will remain -inf.
3. Roughly how long a sequence does it take, with typical probabilities around 0.25, to underflow a 64-bit float?
    * A 64-bit float can only get down to about 10⁻³⁰⁸. Since 0.25ⁿ reaches that around n ≈ 500, multiplying a few hundred probabilities underflows to 0. Real DNA sequences are much longer, so we use logs, which just become large negative numbers instead.

In [12]:
# Import the libraries needed for our code
import numpy as np
import pandas as pd
import math

In [13]:
# Convert probabilities into log probabilities to avoid numerical underflow
# Since log(0) is undefined, we return negative infinity for a probability of 0
# This also ensures that impossible paths are not selected by the algorithm

def _log(probability):
    if probability == 0:
        return -math.inf
    return math.log(probability)

In [ ]:
class HiddenMarkov:
    def __init__(self, init_probs, trans_probs, emit_probs):
        """Store the HMM probabilities. The hidden states come from the keys of init_probs."""
        self.states = tuple(init_probs)
        self.init_probs = init_probs
        self.trans_probs = trans_probs
        self.emit_probs = emit_probs

    def validation(self, observation):
        """Check the observation and the probability tables before running Viterbi."""
        # Nothing to decode if the sequence is empty
        if observation == "":
            return [], 0.0
                
        # Collect every symbol the model can emit (the alphabet)
        alphabet = set()    
        for emissions in self.emit_probs.values():
            alphabet.update(emissions.keys())

            # Each state's emission probabilities should add up to 1
            if round(sum(emissions.values()), 6) != 1:
                raise ValueError("Emission probabilities must sum to 1")

        # The starting probabilities should add up to 1
        if round(sum(self.init_probs.values()), 6) != 1:
            raise ValueError("Initial probabilities must sum to 1")

        # Each state's transition probabilities should add up to 1
        for transitions in self.trans_probs.values():
            if round(sum(transitions.values()), 6) != 1:
                raise ValueError("Transition probabilities must sum to 1")

        # Every observed symbol must be one the model can emit
        for symbol in observation:
            if symbol not in alphabet:
                raise ValueError(f"Invalid observation: {symbol}")
        
    def _initialization(self, observation):
        """Step 1: build the score and backpointer tables and fill in the first position."""

        # create score table and previous state tables
        # One column per observation; scores start at -inf, backpointers at None
        scores = []
        previous = []

        for symbol in observation:
            score_column = {}
            previous_column = {}

            for state in self.states:
                score_column[state] = float("-inf")
                previous_column[state] = None
            
            scores.append(score_column)
            previous.append(previous_column)

        # First position: log(start in state) + log(state emits first symbol)
        for state in self.states:
            scores[0][state] = (
            _log(self.init_probs[state]) 
            + _log(self.emit_probs[state][observation[0]])
            )
        
        return scores, previous
    

    def _recursion(self, observation):
        """Step 2: fill in the best score and backpointer for every state at every position."""

        scores, previous = self._initialization(observation)

        # Try every preceding state for each current state and keep the best one
        for position in range(1, len(observation)):
            for current in self.states:
                for preceding in self.states:
                    # Previous best score + transition + emission
                    candidate = (
                        scores[position - 1][preceding]
                        + _log(self.trans_probs[preceding][current])
                        + _log(self.emit_probs[current][observation[position]])
                    )
                    # Save the better score and remember where it came from
                    if candidate > scores[position][current]:
                        scores[position][current] = candidate
                        previous[position][current] = preceding
        return scores, previous
    


    def traceback(self, observation):
        """Steps 3 and 4: pick the best final state, then traceback to get the path."""

        # Return an empty path if there are no observations
        if not observation:
            return [], 0.0

        # Check that our observations and probabilities are valid
        self.validation(observation)

        scores, previous = self._recursion(observation)

        # Termination: the highest score in the last column ends the best path
        final = max(scores[-1], key=scores[-1].get)
        best_log = scores[-1][final]

        # -inf means every possible path had a probability of 0
        if best_log == -math.inf:
            raise ValueError("No valid path can produce the observations.")

        # Traceback: start at the final state and step backwards
        path = [final]

        for position in range(len(observation) - 1, 0, -1):
            preceding_state = previous[position][path[-1]]
            path.append(preceding_state)

        # We built the path backwards, so flip it
        path.reverse()

        return path, best_log

In [15]:
print(HiddenMarkov.__dict__.keys())

dict_keys(['__module__', '__firstlineno__', '__init__', 'validation', '_initialization', '_recursion', 'traceback', '__static_attributes__', '__dict__', '__weakref__', '__doc__'])


In [16]:
# EXAMPLE INPUTS (provided with the assignment)
# Two hidden states: "I" = CpG island, "G" = genomic background.
# These are example values for testing, not real genomic data.

# Example observation sequence
obs = "GGCACTGAA"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.2,
    "G": 0.8
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.7, "G": 0.3},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

In [17]:
# RUN THE MODEL

# Build the HMM from the example probabilities, check the inputs,
# then decode the observation sequence to get the most likely path.
model = HiddenMarkov(init_probs, trans_probs, emit_probs)

model.validation(obs)
best_path, best_log = model.traceback(obs)

print("Best path:", best_path)
print("Best log probability:", best_log)

Best path: ['G', 'G', 'G', 'G', 'G', 'G', 'G', 'G', 'G']
Best log probability: -13.929108456051065


In [18]:
# Tests below provided by Claude
# Expected: all G's, score about -13.929108
model = HiddenMarkov(init_probs, trans_probs, emit_probs)
model.validation(obs)
model.traceback(obs)

(['G', 'G', 'G', 'G', 'G', 'G', 'G', 'G', 'G'], -13.929108456051065)

![](assets/rotation.png)

In [19]:
# Add up the log probabilities for the all-G path ourselves.
# Expected: this number should match the score from Test 1
hand_score = math.log(init_probs["G"]) + math.log(emit_probs["G"][obs[0]])

for symbol in obs[1:]:
    hand_score += math.log(trans_probs["G"]["G"]) + math.log(emit_probs["G"][symbol])

hand_score

-13.929108456051067

In [20]:
# I: 0.2 * 0.4 = 0.08,  G: 0.8 * 0.2 = 0.16
# Expected: ['G'], score = log(0.16), about -1.832581
model.traceback("C")

(['G'], -1.83258146374831)

In [21]:
# Expected: ([], 0.0)
model.validation("")

([], 0.0)

In [22]:
# Testing Viterbi algorithm
hmm = HiddenMarkov(init_probs, trans_probs, emit_probs)

# Test 1: Professor's example
path, score = hmm.traceback("GGCACTGAA")
assert path == ["G"] * 9
assert math.isclose(score, -13.929108456051065)

# Test 2: Single observation
path, score = hmm.traceback("G")
assert path == ["G"]
assert math.isclose(score, math.log(0.16))

# Test 3: Different observation sequence
path, score = hmm.traceback("AAAA")
assert len(path) == 4
assert all(state in hmm.states for state in path)
assert math.isfinite(score)

print("All tests passed!")

All tests passed!


![](assets/rotation.png)